# SegOCR — Generate 5-Worker Ensemble Dataset (Kaggle)

Generates 50,000 synthetic samples (5 worker slices × 10K each) at 512², sized to fit Kaggle's 20 GB `/kaggle/working/` cap (~18 GB total at ~365 KB/sample). Publishes as a Kaggle Dataset for the 5 parallel training notebooks.

**Before running:**
1. Settings → Accelerator: **None** (CPU is enough; generation is not GPU-bound).
2. Settings → Persistence: **Files only** is fine.
3. Click **Save Version → Save & Run All**.

**Wall time:** ~3–4 hours on Kaggle CPU.

**After it finishes:** click *New Dataset* from the output panel and publish as `segocr-ensemble-50k` (Public). The 5 training notebooks attach this dataset by that slug.

## 1 / Setup — clone repo + install deps

In [ ]:
import os
if not os.path.isdir('/kaggle/working/segocr'):
    !git clone https://github.com/mauryantitans/SegOCR.git /kaggle/working/segocr
%cd /kaggle/working/segocr
!git pull --quiet
!pip install -q -e .
!pip install -q -r requirements/base.txt

## 2 / Verify font availability + plan

In [ ]:
import os
NUM_WORKERS = 5
SAMPLES_PER_WORKER = 10_000   # 5 × 10K = 50K total, ~18 GB on disk
                              # (Kaggle's /kaggle/working/ cap is 20 GB)
DATASET_OUTPUT = '/kaggle/working/segocr-ensemble-50k'
os.makedirs(DATASET_OUTPUT, exist_ok=True)

n_fonts = sum(
    1 for root, _, files in os.walk('/usr/share/fonts')
    for f in files if f.endswith(('.ttf', '.otf'))
)
print(f'System fonts available: {n_fonts}')
print(f'Output: {DATASET_OUTPUT}')
print(f'Plan:   {NUM_WORKERS} slices × {SAMPLES_PER_WORKER} = {NUM_WORKERS * SAMPLES_PER_WORKER} samples total')

## 3 / Build the generator config

In [ ]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(Path('segocr/configs/default.yaml').read_text())

cfg['generator']['fonts']['root_dir'] = '/usr/share/fonts'
cfg['generator']['fonts']['cache_path'] = '/kaggle/working/font_cache.json'
cfg['generator']['fonts']['min_size'] = 40
cfg['generator']['fonts']['max_size'] = 128
cfg['generator']['image_size'] = [512, 512]
cfg['generator']['num_workers'] = 2

cfg['generator']['text']['min_length'] = 2
cfg['generator']['text']['max_length'] = 20
cfg['generator']['text']['min_words_per_line'] = 1
cfg['generator']['text']['max_words_per_line'] = 3
cfg['generator']['text']['max_lines'] = 1
cfg['generator']['text']['case_distribution'] = {
    'lower': 0.50, 'upper': 0.20, 'mixed': 0.20, 'title': 0.10,
}
cfg['generator']['text']['rare_char_boost'] = 4.0
cfg['generator']['text']['corpus_paths'] = [
    {'path': 'BUNDLED:signs', 'tag': 'signs', 'weight': 0.30},
    {'path': 'BUNDLED:receipts', 'tag': 'receipts', 'weight': 0.20},
    {'path': 'BUNDLED:names', 'tag': 'names', 'weight': 0.20},
    {'path': 'BUNDLED:numbers', 'tag': 'numbers', 'weight': 0.30},
]
cfg['generator']['layout']['modes'] = {
    'horizontal': 0.50, 'rotated': 0.20, 'curved': 0.10,
    'perspective': 0.10, 'deformed': 0.10, 'paragraph': 0.0,
}
cfg['generator']['background']['natural_image_dirs'] = []
cfg['generator']['background']['tier_distribution'] = {
    'tier1_solid': 0.40, 'tier2_procedural': 0.30,
    'tier3_natural': 0.25, 'tier4_adversarial': 0.05,
}
cfg['generator']['compositing']['color_strategy'] = {
    'contrast_aware': 0.60, 'random': 0.30, 'low_contrast': 0.10,
}
cfg['generator']['degradation']['blur'] = {
    'probability': 0.30, 'gaussian_sigma': [0.3, 1.0],
    'motion_kernel': [3, 7], 'defocus_radius': [1, 3],
}
cfg['generator']['degradation']['noise']['probability'] = 0.40
cfg['generator']['degradation']['noise']['gaussian_sigma'] = [5, 20]
cfg['generator']['degradation']['occlusion']['probability'] = 0.05

config_path = '/kaggle/working/gen_config.yaml'
Path(config_path).write_text(yaml.safe_dump(cfg))
print(f'Config: {config_path}')

## 4 / Generate 5 worker slices serially

Each slice gets a deterministic, disjoint range of indices (worker N uses indices `N * SAMPLES_PER_WORKER .. (N+1) * SAMPLES_PER_WORKER - 1`). Same code + same indices = same images, always — so any of the 5 training accounts will see exactly the data this notebook produces here.

Generation runs ~5–10 img/s on Kaggle CPU at 512². Each 10K slice takes ~20–35 min. Total: ~2–3 hours.

In [ ]:
import time
for worker_id in range(NUM_WORKERS):
    output = f'{DATASET_OUTPUT}/worker{worker_id}'
    if os.path.isdir(f'{output}/images') and len(os.listdir(f'{output}/images')) >= SAMPLES_PER_WORKER:
        print(f'[worker {worker_id}] already has {SAMPLES_PER_WORKER} samples — skipping')
        continue
    offset = worker_id * SAMPLES_PER_WORKER
    print(f'\n=== Generating worker {worker_id}: indices {offset}..{offset + SAMPLES_PER_WORKER - 1} ===')
    t0 = time.time()
    !python -m scripts.generate_dataset --config {config_path} --num-images {SAMPLES_PER_WORKER} --output {output} --index-offset {offset}
    print(f'[worker {worker_id}] done in {(time.time() - t0)/60:.1f} min')

print('\nAll 5 worker slices generated.')

## 5 / Sanity-check output layout

In [ ]:
for worker_id in range(NUM_WORKERS):
    worker_dir = f'{DATASET_OUTPUT}/worker{worker_id}'
    n_imgs = len(os.listdir(f'{worker_dir}/images')) if os.path.isdir(f'{worker_dir}/images') else 0
    n_sem  = len(os.listdir(f'{worker_dir}/semantic')) if os.path.isdir(f'{worker_dir}/semantic') else 0
    n_inst = len(os.listdir(f'{worker_dir}/instance')) if os.path.isdir(f'{worker_dir}/instance') else 0
    n_meta = len(os.listdir(f'{worker_dir}/metadata')) if os.path.isdir(f'{worker_dir}/metadata') else 0
    print(f'worker{worker_id}: {n_imgs} images / {n_sem} semantic / {n_inst} instance / {n_meta} metadata')

## 6 / Publish as a Kaggle Dataset

1. Click **Save Version → Save & Run All** at the top right (if you haven't already).
2. When the run completes, the **Output** tab shows the `segocr-ensemble-50k/` folder.
3. Click the **New Dataset** button in the Output panel.
4. Slug: `segocr-ensemble-50k`. Visibility: **Public** (simplest for cross-account sharing).
5. After publish, copy the dataset URL — each training notebook attaches it via *Add Data*.

Total dataset size: ~18 GB (under Kaggle's 100 GB per-dataset limit). The /kaggle/working/ cap during generation is what constrains us, not the dataset hosting size.